In [0]:
#1. leemos el archivo JSON usando "DataFrameReader" de Spark

# Importamos las librerias que se van a utilizar
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define el la estructura personName
movie_genre_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("genreId", IntegerType(), True)
])

movie_genre_df = spark.read\
    .schema(movie_genre_schema)\
    .json("abfss://bronze@lsdata01.dfs.core.windows.net/movie_genre.json")

display(movie_genre_df)


In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
from pyspark.sql.functions import  current_timestamp, lit

movie_genre_final_df = movie_genre_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("genreId", "genre_id")\
    .withColumn("ingestion_date", current_timestamp())\
    .withColumn("enviroment", lit("Produccion"))

display(movie_genre_final_df)

In [0]:
#Paso 3 - Guardar datos en datalake en formato parket y particionado por Movie_id

movie_genre_final_df.write.mode("overwrite").partitionBy("movie_id").parquet("abfss://silver@lsdata01.dfs.core.windows.net/movie_genre")



In [0]:
df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/movie_genre")
display(df)
